## Лабораторная работа № 1 
## Выполнение разведочного анализа больших данных с использованием фреймворка Apache Spark

### Часть 1

В данной части работы рассмотрены:
* загрузка данных из HDFS;
* базовые преобразования данных;
* загрузка преобразованных данных в таблицу `Apache Airflow`.

Подключим необходимые библиотеки.

In [1]:
import os
from pyspark.sql import SparkSession, DataFrame
from pyspark import SparkConf
from pyspark.sql.functions import (
    regexp_replace,
    regexp_extract_all,
    regexp_extract,
    col,
    lit,
    when,
    to_date,
    from_unixtime,
    split,
    trim,
    year
)

Сформируем объект конфигурации для `Apache Spark`, указав необходимые параметры.

In [2]:
def create_spark_configuration() -> SparkConf:
    """
    Создает и конфигурирует экземпляр SparkConf для приложения Spark.

    Returns:
        SparkConf: Настроенный экземпляр SparkConf.
    """
    # Получаем имя пользователя
    # Получаем имя пользователя
    user_name = os.getenv("USER")
    
    conf = SparkConf()
    conf.setAppName("Lab_1_DC")
    conf.setMaster("local[*]")
    # conf.set("spark.submit.deployMode", "client")
    conf.set("spark.executor.memory", "12g")
    conf.set("spark.executor.cores", "6")
    conf.set("spark.executor.instances", "1")
    conf.set("spark.driver.memory", "4g")
    conf.set("spark.driver.cores", "1")

    return conf

Создаём сам объект конфигурации.

In [3]:
conf = create_spark_configuration()

Создаём и выводим на экран сессию `Apache Spark`. В процессе создания сессии происходит подключение к кластеру `Apache Hadoop`, что может занять некоторое время.

In [4]:
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark

Для исследования будем использовать датасет `"Big Sales Data"`, расположенный на платформе `Kaggle` по адресу https://www.kaggle.com/datasets/pigment/big-sales-data/data.

Указываем путь.

In [5]:
test_data_name = 'test_data'
train_data_name = 'train_data'
test_path = f'../data/{test_data_name}/*.pq'
train_path = f'../data/{train_data_name}/*.pq'


Заполняем датафрейм данными из файла.

In [6]:
test_df = spark.read.parquet(test_path)
train_df = spark.read.parquet(train_path)

Выводим фрагмент датафрейма на экран.

In [7]:
test_df.limit(20).toPandas().style\
    .set_properties(**{'text-align': 'left', 'max-width': '0', 'white-space': 'nowrap', 'overflow': 'hidden', 'text-overflow': 'ellipsis'})\
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])\
    .format(precision=2)

,id,rn,pre_since_opened,pre_since_confirmed,pre_pterm,pre_fterm,pre_till_pclose,pre_till_fclose,pre_loans_credit_limit,pre_loans_next_pay_summ,pre_loans_outstanding,pre_loans_total_overdue,pre_loans_max_overdue_sum,pre_loans_credit_cost_rate,pre_loans5,pre_loans530,pre_loans3060,pre_loans6090,pre_loans90,is_zero_loans5,is_zero_loans530,is_zero_loans3060,is_zero_loans6090,is_zero_loans90,pre_util,pre_over2limit,pre_maxover2limit,is_zero_util,is_zero_over2limit,is_zero_maxover2limit,enc_paym_0,enc_paym_1,enc_paym_2,enc_paym_3,enc_paym_4,enc_paym_5,enc_paym_6,enc_paym_7,enc_paym_8,enc_paym_9,enc_paym_10,enc_paym_11,enc_paym_12,enc_paym_13,enc_paym_14,enc_paym_15,enc_paym_16,enc_paym_17,enc_paym_18,enc_paym_19,enc_paym_20,enc_paym_21,enc_paym_22,enc_paym_23,enc_paym_24,enc_loans_account_holder_type,enc_loans_credit_status,enc_loans_credit_type,enc_loans_account_cur,pclose_flag,fclose_flag
0,3000000,1,11,5,17,14,12,11,3,2,3,0,2,2,6,16,5,4,8,1,0,1,1,1,16,2,17,1,1,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,3,3,3,4,3,3,3,4,1,3,4,1,0,0
1,3000000,2,19,16,15,9,12,11,16,3,2,0,2,8,6,16,5,4,8,1,1,1,1,1,15,2,17,0,1,1,0,0,0,0,3,3,3,3,3,3,3,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,2,4,1,0,0
2,3000001,1,16,17,8,5,4,9,5,2,3,0,2,4,6,16,5,4,8,1,0,0,1,1,16,2,17,1,1,1,0,2,1,0,1,0,0,3,3,3,3,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,4,1,0,0
3,3000001,2,16,7,9,0,4,9,1,2,3,0,2,4,6,16,5,4,8,1,0,0,1,1,16,2,17,1,1,1,0,2,1,2,2,2,2,1,0,0,0,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,4,1,0,0
4,3000001,3,10,0,14,7,11,12,2,2,3,0,2,4,3,16,5,4,8,0,1,0,0,1,16,2,17,1,1,1,0,0,1,1,1,1,1,1,3,2,1,2,1,1,1,1,1,1,0,1,2,1,1,0,2,1,3,4,1,0,0
5,3000001,4,10,16,6,8,10,11,2,0,1,0,2,4,6,16,5,4,8,0,1,0,1,1,2,2,17,0,1,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,1,1,2,0,1,0,1
6,3000001,5,0,9,7,6,13,3,11,5,3,0,1,5,6,16,5,4,8,1,0,1,1,1,16,2,0,1,1,0,0,1,1,1,1,0,0,3,3,3,3,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,4,1,0,0
7,3000001,6,13,0,14,7,7,15,15,2,3,0,2,4,5,16,5,4,8,0,1,0,1,1,16,2,17,1,1,1,1,1,1,0,1,1,1,1,1,1,0,1,0,1,1,1,1,1,1,1,1,1,0,1,2,1,3,4,1,0,0
8,3000001,7,18,9,15,9,5,7,1,1,3,0,1,0,6,16,5,4,8,1,0,1,1,1,16,2,3,1,1,0,0,0,0,0,0,0,0,0,0,0,0,2,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,4,1,0,0
9,3000001,8,5,11,7,6,16,12,9,2,3,0,1,7,6,16,5,4,8,1,0,0,1,1,16,2,15,1,1,0,1,2,1,1,0,0,3,3,3,3,3,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,5,1,0,0


In [8]:
train_df.limit(20).toPandas().style\
    .set_properties(**{'text-align': 'left', 'max-width': '0', 'white-space': 'nowrap', 'overflow': 'hidden', 'text-overflow': 'ellipsis'})\
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])\
    .format(precision=2)

,id,rn,pre_since_opened,pre_since_confirmed,pre_pterm,pre_fterm,pre_till_pclose,pre_till_fclose,pre_loans_credit_limit,pre_loans_next_pay_summ,pre_loans_outstanding,pre_loans_total_overdue,pre_loans_max_overdue_sum,pre_loans_credit_cost_rate,pre_loans5,pre_loans530,pre_loans3060,pre_loans6090,pre_loans90,is_zero_loans5,is_zero_loans530,is_zero_loans3060,is_zero_loans6090,is_zero_loans90,pre_util,pre_over2limit,pre_maxover2limit,is_zero_util,is_zero_over2limit,is_zero_maxover2limit,enc_paym_0,enc_paym_1,enc_paym_2,enc_paym_3,enc_paym_4,enc_paym_5,enc_paym_6,enc_paym_7,enc_paym_8,enc_paym_9,enc_paym_10,enc_paym_11,enc_paym_12,enc_paym_13,enc_paym_14,enc_paym_15,enc_paym_16,enc_paym_17,enc_paym_18,enc_paym_19,enc_paym_20,enc_paym_21,enc_paym_22,enc_paym_23,enc_paym_24,enc_loans_account_holder_type,enc_loans_credit_status,enc_loans_credit_type,enc_loans_account_cur,pclose_flag,fclose_flag
0,0,1,18,9,2,3,16,10,11,3,3,0,2,11,6,16,5,4,8,1,1,1,1,1,16,2,17,1,1,1,0,0,3,3,3,3,3,3,3,3,3,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,4,1,0,0
1,0,2,18,9,14,14,12,12,0,3,3,0,2,11,6,16,5,4,8,1,1,1,1,1,16,2,17,1,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,4,1,3,4,1,0,0
2,0,3,18,9,4,8,1,11,11,0,5,0,2,8,6,16,5,4,8,1,1,1,1,1,15,2,17,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,4,1,2,3,1,1,1
3,0,4,4,1,9,12,16,7,12,2,3,0,2,4,6,16,5,4,8,0,1,1,1,1,16,2,17,1,1,1,1,0,0,0,0,0,0,0,0,0,0,1,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,1,1,0,0
4,0,5,5,12,15,2,11,12,10,2,3,0,2,4,6,16,5,4,8,1,1,1,1,1,16,2,17,1,1,1,0,0,0,0,0,0,0,3,3,3,3,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,4,1,0,0
5,0,6,5,0,11,8,12,11,4,2,3,0,2,4,6,16,5,4,8,1,1,1,1,1,9,5,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,3,4,3,3,3,4,1,2,3,1,0,1
6,0,7,3,9,1,2,12,14,15,5,3,0,2,3,6,16,5,4,8,1,1,1,1,1,16,2,17,1,1,1,0,0,0,0,0,0,0,0,3,3,3,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,4,1,0,0
7,0,8,2,9,2,3,12,14,15,5,3,0,2,13,6,16,5,4,8,1,1,1,1,1,16,2,17,1,1,1,0,0,3,3,3,3,3,3,3,3,3,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,4,1,0,0
8,0,9,1,9,11,13,14,8,2,5,1,0,2,11,6,16,5,4,8,1,1,1,1,1,1,2,17,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,3,3,3,3,3,3,4,3,3,3,4,1,2,4,1,0,0
9,0,10,7,9,2,10,8,8,16,4,2,0,2,11,6,16,5,4,8,1,1,1,1,1,15,2,17,0,1,1,0,0,0,0,0,0,3,3,3,3,3,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,2,4,1,0,0


Очевидно, что в целях сохранения ясности изложения и сокращения расчетного времени имеет смысл рассматривать не все столбцы датасета. Оставим следующие колонки, удалив остальные:

**Название столбца** | **Важность** | **Как использовать в train**
---|---|---
**id** | 🔧 Служебный | Идентификатор клиента. Используется для агрегации и связи с `train_target.csv` где находится `flag`
**rn** | ⭐⭐ | Порядковый номер кредита. При агрегации → `count` = количество кредитов у клиента
**pre_loans90** | ⭐⭐⭐⭐⭐ | **Суммировать** по клиенту. `>0` = красный флаг, лучший предиктор дефолта
**pre_loans_total_overdue** | ⭐⭐⭐⭐⭐ | **Сумма** по клиенту. `>0` = клиент уже в текущем дефолте
**pre_loans3060** | ⭐⭐⭐⭐ | **Суммировать** по клиенту. Показатель хронических проблем с платежами
**pre_maxover2limit** | ⭐⭐⭐⭐ | **Максимум** по клиенту. Показывает "дно" финансовой ямы в истории
**pre_util** | ⭐⭐⭐⭐ | **Среднее** и **максимум** по клиенту. `>0.8` = высокая кредитная нагрузка
**pre_over2limit** | ⭐⭐⭐⭐ | **Среднее** по клиенту. Острота текущих проблем относительно лимита
**pre_loans530** | ⭐⭐⭐ | **Суммировать** по клиенту. Частые мелкие просрочки
**pre_loans_outstanding** | ⭐⭐⭐ | **Сумма** по клиенту. Общий объем задолженности
**pre_loans_credit_limit** | ⭐⭐⭐ | **Сумма** по клиенту. Для расчета `pre_util` и общего лимита
**pre_loans_credit_cost_rate** | ⭐⭐⭐ | **Среднее** по клиенту. Дороговизна кредитов (дороже = рискованнее)
**is_zero_loans90** | ⭐⭐ | **Минимум** по клиенту. Если все `=1` → никогда не было серьезных просрочек
**pclose_flag** | ⭐⭐ | **Максимум** по клиенту. Если `=1` → есть кредит без плановой даты закрытия
**fclose_flag** | ⭐⭐ | **Максимум** по клиенту. Если `=1` → есть кредит без фактической даты закрытия при агрегации)

In [9]:
train_df = train_df.select(
    "id",
    "rn",
    "pre_loans90",
    "pre_loans_total_overdue", 
    "pre_loans3060",
    "pre_maxover2limit",
    "pre_util",
    "pre_over2limit",
    "pre_loans530",
    "pre_loans_outstanding",
    "pre_loans_credit_limit",
    "pre_loans_credit_cost_rate",
    "is_zero_loans90",
    "pclose_flag",
    "fclose_flag"
)

test_df = test_df.select(
    "id",
    "rn", 
    "pre_loans90",
    "pre_loans_total_overdue",
    "pre_loans3060",
    "pre_maxover2limit",
    "pre_util",
    "pre_over2limit",
    "pre_loans530",
    "pre_loans_outstanding",
    "pre_loans_credit_limit",
    "pre_loans_credit_cost_rate",
    "is_zero_loans90",
    "pclose_flag",
    "fclose_flag"
)

In [10]:
test_df.limit(20).toPandas().style\
    .set_properties(**{'text-align': 'left', 'max-width': '0', 'white-space': 'nowrap', 'overflow': 'hidden', 'text-overflow': 'ellipsis'})\
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])\
    .format(precision=2)

,id,rn,pre_loans90,pre_loans_total_overdue,pre_loans3060,pre_maxover2limit,pre_util,pre_over2limit,pre_loans530,pre_loans_outstanding,pre_loans_credit_limit,pre_loans_credit_cost_rate,is_zero_loans90,pclose_flag,fclose_flag
0,3000000,1,8,0,5,17,16,2,16,3,3,2,1,0,0
1,3000000,2,8,0,5,17,15,2,16,2,16,8,1,0,0
2,3000001,1,8,0,5,17,16,2,16,3,5,4,1,0,0
3,3000001,2,8,0,5,17,16,2,16,3,1,4,1,0,0
4,3000001,3,8,0,5,17,16,2,16,3,2,4,1,0,0
5,3000001,4,8,0,5,17,2,2,16,1,2,4,1,0,1
6,3000001,5,8,0,5,0,16,2,16,3,11,5,1,0,0
7,3000001,6,8,0,5,17,16,2,16,3,15,4,1,0,0
8,3000001,7,8,0,5,3,16,2,16,3,1,0,1,0,0
9,3000001,8,8,0,5,15,16,2,16,3,9,7,1,0,0


In [11]:
train_df.limit(20).toPandas().style\
    .set_properties(**{'text-align': 'left', 'max-width': '0', 'white-space': 'nowrap', 'overflow': 'hidden', 'text-overflow': 'ellipsis'})\
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])\
    .format(precision=2)

,id,rn,pre_loans90,pre_loans_total_overdue,pre_loans3060,pre_maxover2limit,pre_util,pre_over2limit,pre_loans530,pre_loans_outstanding,pre_loans_credit_limit,pre_loans_credit_cost_rate,is_zero_loans90,pclose_flag,fclose_flag
0,0,1,8,0,5,17,16,2,16,3,11,11,1,0,0
1,0,2,8,0,5,17,16,2,16,3,0,11,1,0,0
2,0,3,8,0,5,17,15,2,16,5,11,8,1,1,1
3,0,4,8,0,5,17,16,2,16,3,12,4,1,0,0
4,0,5,8,0,5,17,16,2,16,3,10,4,1,0,0
5,0,6,8,0,5,4,9,5,16,3,4,4,1,0,1
6,0,7,8,0,5,17,16,2,16,3,15,3,1,0,0
7,0,8,8,0,5,17,16,2,16,3,15,13,1,0,0
8,0,9,8,0,5,17,1,2,16,1,2,11,1,0,0
9,0,10,8,0,5,17,15,2,16,2,16,11,1,0,0


Выведем на экран метаданные датасета.

In [12]:
test_df.printSchema()

root
 |-- id: long (nullable = true)
 |-- rn: long (nullable = true)
 |-- pre_loans90: long (nullable = true)
 |-- pre_loans_total_overdue: long (nullable = true)
 |-- pre_loans3060: long (nullable = true)
 |-- pre_maxover2limit: long (nullable = true)
 |-- pre_util: long (nullable = true)
 |-- pre_over2limit: long (nullable = true)
 |-- pre_loans530: long (nullable = true)
 |-- pre_loans_outstanding: long (nullable = true)
 |-- pre_loans_credit_limit: long (nullable = true)
 |-- pre_loans_credit_cost_rate: long (nullable = true)
 |-- is_zero_loans90: long (nullable = true)
 |-- pclose_flag: long (nullable = true)
 |-- fclose_flag: long (nullable = true)



In [13]:
train_df.printSchema()

root
 |-- id: long (nullable = true)
 |-- rn: long (nullable = true)
 |-- pre_loans90: long (nullable = true)
 |-- pre_loans_total_overdue: long (nullable = true)
 |-- pre_loans3060: long (nullable = true)
 |-- pre_maxover2limit: long (nullable = true)
 |-- pre_util: long (nullable = true)
 |-- pre_over2limit: long (nullable = true)
 |-- pre_loans530: long (nullable = true)
 |-- pre_loans_outstanding: long (nullable = true)
 |-- pre_loans_credit_limit: long (nullable = true)
 |-- pre_loans_credit_cost_rate: long (nullable = true)
 |-- is_zero_loans90: long (nullable = true)
 |-- pclose_flag: long (nullable = true)
 |-- fclose_flag: long (nullable = true)



Видно, что все столбцы датасета содержат long тип данных, что не соответствует ожиданиям. Выполним преобразования типов данных некоторых столбцов.

In [15]:
def transform_dataframe(data: DataFrame) -> DataFrame:
    """
    Оптимизирует типы данных для выбранных 15 колонок кредитного датафрейма.
    
    Args:
        data (DataFrame): DataFrame с 15 выбранными колонками.
    
    Returns:
        DataFrame: Оптимизированный DataFrame с правильными типами данных.
    """
    # Все преобразования для ваших 15 колонок
    data = data \
        .withColumn("id", col("id").cast("int")) \
        .withColumn("rn", col("rn").cast("short")) \
        .withColumn("is_zero_loans90", col("is_zero_loans90").cast("byte")) \
        .withColumn("pclose_flag", col("pclose_flag").cast("byte")) \
        .withColumn("fclose_flag", col("fclose_flag").cast("byte")) \
        .withColumn("pre_loans90", col("pre_loans90").cast("short")) \
        .withColumn("pre_loans3060", col("pre_loans3060").cast("short")) \
        .withColumn("pre_loans530", col("pre_loans530").cast("short")) \
        .withColumn("pre_loans_total_overdue", col("pre_loans_total_overdue").cast("float")) \
        .withColumn("pre_loans_outstanding", col("pre_loans_outstanding").cast("float")) \
        .withColumn("pre_loans_credit_limit", col("pre_loans_credit_limit").cast("float")) \
        .withColumn("pre_util", col("pre_util").cast("float")) \
        .withColumn("pre_over2limit", col("pre_over2limit").cast("float")) \
        .withColumn("pre_maxover2limit", col("pre_maxover2limit").cast("float")) \
        .withColumn("pre_loans_credit_cost_rate", col("pre_loans_credit_cost_rate").cast("float"))
    
    return data


In [16]:
test_df = transform_dataframe(test_df)
train_df = transform_dataframe(train_df)

In [17]:
test_df.limit(20).toPandas().style\
    .set_properties(**{'text-align': 'left', 'max-width': '0', 'white-space': 'nowrap', 'overflow': 'hidden', 'text-overflow': 'ellipsis'})\
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])\
    .format(precision=2)

,id,rn,pre_loans90,pre_loans_total_overdue,pre_loans3060,pre_maxover2limit,pre_util,pre_over2limit,pre_loans530,pre_loans_outstanding,pre_loans_credit_limit,pre_loans_credit_cost_rate,is_zero_loans90,pclose_flag,fclose_flag
0,3000000,1,8,0.00,5,17.00,16.00,2.00,16,3.00,3.00,2.00,1,0,0
1,3000000,2,8,0.00,5,17.00,15.00,2.00,16,2.00,16.00,8.00,1,0,0
2,3000001,1,8,0.00,5,17.00,16.00,2.00,16,3.00,5.00,4.00,1,0,0
3,3000001,2,8,0.00,5,17.00,16.00,2.00,16,3.00,1.00,4.00,1,0,0
4,3000001,3,8,0.00,5,17.00,16.00,2.00,16,3.00,2.00,4.00,1,0,0
5,3000001,4,8,0.00,5,17.00,2.00,2.00,16,1.00,2.00,4.00,1,0,1
6,3000001,5,8,0.00,5,0.00,16.00,2.00,16,3.00,11.00,5.00,1,0,0
7,3000001,6,8,0.00,5,17.00,16.00,2.00,16,3.00,15.00,4.00,1,0,0
8,3000001,7,8,0.00,5,3.00,16.00,2.00,16,3.00,1.00,0.00,1,0,0
9,3000001,8,8,0.00,5,15.00,16.00,2.00,16,3.00,9.00,7.00,1,0,0


In [18]:
train_df.limit(20).toPandas().style\
    .set_properties(**{'text-align': 'left', 'max-width': '0', 'white-space': 'nowrap', 'overflow': 'hidden', 'text-overflow': 'ellipsis'})\
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])\
    .format(precision=2)

,id,rn,pre_loans90,pre_loans_total_overdue,pre_loans3060,pre_maxover2limit,pre_util,pre_over2limit,pre_loans530,pre_loans_outstanding,pre_loans_credit_limit,pre_loans_credit_cost_rate,is_zero_loans90,pclose_flag,fclose_flag
0,0,1,8,0.00,5,17.00,16.00,2.00,16,3.00,11.00,11.00,1,0,0
1,0,2,8,0.00,5,17.00,16.00,2.00,16,3.00,0.00,11.00,1,0,0
2,0,3,8,0.00,5,17.00,15.00,2.00,16,5.00,11.00,8.00,1,1,1
3,0,4,8,0.00,5,17.00,16.00,2.00,16,3.00,12.00,4.00,1,0,0
4,0,5,8,0.00,5,17.00,16.00,2.00,16,3.00,10.00,4.00,1,0,0
5,0,6,8,0.00,5,4.00,9.00,5.00,16,3.00,4.00,4.00,1,0,1
6,0,7,8,0.00,5,17.00,16.00,2.00,16,3.00,15.00,3.00,1,0,0
7,0,8,8,0.00,5,17.00,16.00,2.00,16,3.00,15.00,13.00,1,0,0
8,0,9,8,0.00,5,17.00,1.00,2.00,16,1.00,2.00,11.00,1,0,0
9,0,10,8,0.00,5,17.00,15.00,2.00,16,2.00,16.00,11.00,1,0,0


In [19]:
test_df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- rn: short (nullable = true)
 |-- pre_loans90: short (nullable = true)
 |-- pre_loans_total_overdue: float (nullable = true)
 |-- pre_loans3060: short (nullable = true)
 |-- pre_maxover2limit: float (nullable = true)
 |-- pre_util: float (nullable = true)
 |-- pre_over2limit: float (nullable = true)
 |-- pre_loans530: short (nullable = true)
 |-- pre_loans_outstanding: float (nullable = true)
 |-- pre_loans_credit_limit: float (nullable = true)
 |-- pre_loans_credit_cost_rate: float (nullable = true)
 |-- is_zero_loans90: byte (nullable = true)
 |-- pclose_flag: byte (nullable = true)
 |-- fclose_flag: byte (nullable = true)



In [20]:
train_df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- rn: short (nullable = true)
 |-- pre_loans90: short (nullable = true)
 |-- pre_loans_total_overdue: float (nullable = true)
 |-- pre_loans3060: short (nullable = true)
 |-- pre_maxover2limit: float (nullable = true)
 |-- pre_util: float (nullable = true)
 |-- pre_over2limit: float (nullable = true)
 |-- pre_loans530: short (nullable = true)
 |-- pre_loans_outstanding: float (nullable = true)
 |-- pre_loans_credit_limit: float (nullable = true)
 |-- pre_loans_credit_cost_rate: float (nullable = true)
 |-- is_zero_loans90: byte (nullable = true)
 |-- pclose_flag: byte (nullable = true)
 |-- fclose_flag: byte (nullable = true)



# Сохранение DataFrame в формате Parquet

### Запись данных из Parquet

In [22]:
database_name = "database"

output_path_test = f"{database_name}/{test_data_name}"
output_path_train = f"{database_name}/{train_data_name}"

print(f"Сохраняем данные в: {output_path_test}")
print(f"Сохраняем данные в: {output_path_train}")

test_df.write.mode("overwrite").option("compression", "snappy").parquet(output_path_test)
train_df.write.mode("overwrite").option("compression", "snappy").parquet(output_path_train)

print("Данные успешно сохранены в Parquet")

Сохраняем данные в: database/test_data
Сохраняем данные в: database/train_data
Данные успешно сохранены в Parquet


### Чтение данных из Parquet

In [23]:
df_parquet = spark.read.parquet(output_path_test)

print("Данные успешно загружены из Parquet")
print(f"Количество строк: {df_parquet.count()}")
print(f"Количество колонок: {len(df_parquet.columns)}")

# Показать схему данных
print("Схема данных:")
df_parquet.printSchema()

# Показать первые несколько строк
print("Первые 10 строк:")

df_parquet.limit(10).toPandas().style\
    .set_properties(**{'text-align': 'left', 'max-width': '0', 'white-space': 'nowrap', 'overflow': 'hidden', 'text-overflow': 'ellipsis'})\
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])\
    .format(precision=2)

Данные успешно загружены из Parquet
Количество строк: 4724601
Количество колонок: 15
Схема данных:
root
 |-- id: integer (nullable = true)
 |-- rn: short (nullable = true)
 |-- pre_loans90: short (nullable = true)
 |-- pre_loans_total_overdue: float (nullable = true)
 |-- pre_loans3060: short (nullable = true)
 |-- pre_maxover2limit: float (nullable = true)
 |-- pre_util: float (nullable = true)
 |-- pre_over2limit: float (nullable = true)
 |-- pre_loans530: short (nullable = true)
 |-- pre_loans_outstanding: float (nullable = true)
 |-- pre_loans_credit_limit: float (nullable = true)
 |-- pre_loans_credit_cost_rate: float (nullable = true)
 |-- is_zero_loans90: byte (nullable = true)
 |-- pclose_flag: byte (nullable = true)
 |-- fclose_flag: byte (nullable = true)

Первые 10 строк:


,id,rn,pre_loans90,pre_loans_total_overdue,pre_loans3060,pre_maxover2limit,pre_util,pre_over2limit,pre_loans530,pre_loans_outstanding,pre_loans_credit_limit,pre_loans_credit_cost_rate,is_zero_loans90,pclose_flag,fclose_flag
0,3000000,1,8,0.00,5,17.00,16.00,2.00,16,3.00,3.00,2.00,1,0,0
1,3000000,2,8,0.00,5,17.00,15.00,2.00,16,2.00,16.00,8.00,1,0,0
2,3000001,1,8,0.00,5,17.00,16.00,2.00,16,3.00,5.00,4.00,1,0,0
3,3000001,2,8,0.00,5,17.00,16.00,2.00,16,3.00,1.00,4.00,1,0,0
4,3000001,3,8,0.00,5,17.00,16.00,2.00,16,3.00,2.00,4.00,1,0,0
5,3000001,4,8,0.00,5,17.00,2.00,2.00,16,1.00,2.00,4.00,1,0,1
6,3000001,5,8,0.00,5,0.00,16.00,2.00,16,3.00,11.00,5.00,1,0,0
7,3000001,6,8,0.00,5,17.00,16.00,2.00,16,3.00,15.00,4.00,1,0,0
8,3000001,7,8,0.00,5,3.00,16.00,2.00,16,3.00,1.00,0.00,1,0,0
9,3000001,8,8,0.00,5,15.00,16.00,2.00,16,3.00,9.00,7.00,1,0,0


После успешной записи таблицы останавливаем сессию `Apache Spark`.

In [24]:
spark.stop()